# Driver Extract Data Processing

In [ ]:
import os
import ast
import random
import numpy as np
import pandas as pd
import networkx as nx

import sys
sys.path.append('../data')

from utils.functions import load_yaml
from data.reference import residue1to3
from utils.graphfunction import load_graph, get_uniprot_from_nodes, get_res_from_nodes, get_pos_from_nodes, save_graph

# 1. Load Configurations
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

## 1. Load Graph and Extract Valid Nodes

In [ ]:
# Load the main structural graph
final_graph = load_graph(f"{DATABASE}/cleaned_weighted_graph_wAtt_v050126.pkl")

# Extract all nodes from the graph's connected components
connected_components = list(nx.connected_components(final_graph))
sorted_components = sorted(connected_components, key=len, reverse=True)

all_nodes = [node for cc in sorted_components for node in cc]
graph_nodes_df = pd.DataFrame({'node_id': all_nodes})

print(f"Total nodes extracted from graph: {len(graph_nodes_df)}")

## 2. Load and Prepare CDS (Coding Sequence) Context

In [ ]:
# Load CDS context data and merge with graph nodes
cds_df = pd.read_csv(f"{DATABASE}/final_nodes_cds_context.csv")
nodes_with_cds = pd.merge(graph_nodes_df, cds_df.iloc[:, :2], on='node_id', how='left')

# Extract residue and position directly from the node_id
nodes_with_cds['res'] = nodes_with_cds['node_id'].apply(get_res_from_nodes)
nodes_with_cds['pos'] = nodes_with_cds['node_id'].apply(get_pos_from_nodes).astype(str)

# Clean and expand ensemble IDs (ensmbl_id contains string representation of lists)
nodes_with_cds['ensmbl_id'] = nodes_with_cds['ensmbl_id'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
nodes_with_cds = nodes_with_cds.explode('ensmbl_id')

# Strip versions from ENST IDs (e.g., ENST00000366667.4 -> ENST00000366667)
nodes_with_cds['ensmbl_id'] = nodes_with_cds['ensmbl_id'].str.split('.').str[0]
nodes_with_cds.dropna(subset=['ensmbl_id'], inplace=True)

nodes_with_cds.head()

## 3. Load and Clean Cancer Mutations Dataset

In [ ]:
# Load and combine cancer mutation datasets
cancer_benchmark_df = pd.read_csv('../data/cancer_mutations_benchmark.csv')
cancer_benchmark_updated_df = pd.read_csv('../data/reference/cancer_mutations_benchmark_updated.csv')

cancer_mutations_df = pd.concat([cancer_benchmark_df, cancer_benchmark_updated_df], axis=0, ignore_index=True)

# Drop rows with missing essential identifiers or missing mutations
cancer_mutations_df.dropna(subset=['Transcript ID', 'Swissprot ID'], axis=0, how='all', inplace=True)
cancer_mutations_df.dropna(subset=['mutation'], axis=0, inplace=True)

# Helper function to convert 1-letter to 3-letter residue codes
def get_residue(mutation):
    first_char = str(mutation)[0].upper()
    res = residue1to3.get(first_char)
    return res.lower() if res is not None else None

# Parse mutation string to get position and residue (e.g., 'V600E' -> pos: 600, res: 'val')
cancer_mutations_df['pos'] = cancer_mutations_df['mutation'].apply(lambda x: str(x)[1:-1])
cancer_mutations_df['pos'] = pd.to_numeric(cancer_mutations_df['pos'], errors='coerce')

cancer_mutations_df['res'] = cancer_mutations_df['mutation'].apply(get_residue)

# Drop invalid positions/residues and enforce string type for consistent matching
cancer_mutations_df.dropna(subset=['res', 'pos'], axis=0, inplace=True)
cancer_mutations_df['pos'] = cancer_mutations_df['pos'].astype(int).astype(str) 

cancer_mutations_df

In [ ]:
cancer_mutations_df.info()

## 4. Match Mutations to Graph Nodes

In [ ]:
# 4a. Process mutations that already have a Swissprot ID
mutations_with_swissprot = cancer_mutations_df[cancer_mutations_df['Swissprot ID'].notna()].copy()

mutations_with_swissprot = mutations_with_swissprot[['Swissprot ID', 'label', 'pos', 'res', 'Transcript ID']]
mutations_with_swissprot.rename(columns={'Swissprot ID': 'uniprot', 'Transcript ID': 'ensmbl_id'}, inplace=True)

mutations_with_swissprot['uniprot'] = mutations_with_swissprot['uniprot'].str.lower()
mutations_with_swissprot['node_id'] = (
    mutations_with_swissprot['uniprot'] + '_' + 
    mutations_with_swissprot['pos'] + '_' + 
    mutations_with_swissprot['res']
)

print(f"Mutations matched directly via Swissprot ID: {len(mutations_with_swissprot)}")

In [ ]:
# 4b. Process remaining mutations by matching Transcript ID (ENST ID) to our graph nodes
cancer_mutations_df['ENST_ID'] = cancer_mutations_df['Transcript ID'].str.extract(r'(ENST\d+)')
mutations_with_enst = cancer_mutations_df.dropna(subset=['ENST_ID']).copy()

# Find ENST IDs that exist in our graph's CDS context
valid_enst_ids = set(nodes_with_cds['ensmbl_id'])
mutations_matched_by_enst = mutations_with_enst[mutations_with_enst['ENST_ID'].isin(valid_enst_ids)].copy()

mutations_matched_by_enst.rename(columns={"ENST_ID": "ensmbl_id"}, inplace=True)

# Merge with nodes_with_cds to construct the correct node_id
matched_enst_df = pd.merge(
    mutations_matched_by_enst[['ensmbl_id', 'pos', 'res', 'label']], 
    nodes_with_cds, 
    on=['res', 'pos', 'ensmbl_id'], 
    how='inner'
)
matched_enst_df['uniprot'] = matched_enst_df['node_id'].apply(get_uniprot_from_nodes)

print(f"Mutations matched via ENST ID mapping: {len(matched_enst_df)}")

## 5. Combine and Deduplicate
- Priority is given to `driver` labels over others during deduplication.

In [ ]:
def prioritize_drivers_and_deduplicate(df):
    return df.assign(is_driver=(df['label'] == 'driver')) \
             .sort_values('is_driver', ascending=False) \
             .drop_duplicates(subset=['node_id'], keep='first') \
             .drop(columns=['is_driver'])

# Deduplicate ENST-matched dataset
dedup_matched_enst_df = prioritize_drivers_and_deduplicate(matched_enst_df)

# Combine both matched datasets
final_cancer_driver_df = pd.concat([dedup_matched_enst_df, mutations_with_swissprot], axis=0, ignore_index=True)

# Final deduplication across the combined dataset
final_cancer_driver_df = prioritize_drivers_and_deduplicate(final_cancer_driver_df)
final_cancer_driver_df

In [ ]:
final_cancer_driver_df[final_cancer_driver_df.duplicated(subset=['node_id'])]

In [ ]:
final_cancer_driver_df.label.value_counts()

In [ ]:
DATABASE

In [ ]:
# Save Final DataFrame
# final_cancer_driver_df.to_csv(f'{DATABASE}/matched_cancer_driver_df.csv', index=False)

# EDA

In [ ]:
SEED = 777
rng = random.Random(SEED)

cc_list = [final_graph.subgraph(c).copy() for c in nx.connected_components(final_graph)]
cc_list.sort(key=lambda x: len(x.nodes()), reverse=True)

num_total = len(cc_list)
train_val_idx = rng.sample(range(num_total), min(124, num_total))
exclude_values = {45, 46}  # incldue cancer driver
train_val_idx = [i for i in train_val_idx if i not in exclude_values]
test_idx = [i for i in range(num_total) if i not in train_val_idx]

val_ratio = 0.20
num_val = int(len(train_val_idx) * val_ratio)
val_idx = rng.sample(train_val_idx, num_val)
train_idx = [i for i in train_val_idx if i not in val_idx]

train_idx.sort()
val_idx.sort()
test_idx.sort()

In [ ]:
len(train_idx), len(val_idx), len(test_idx)

In [ ]:
train_nodes = [node for idx in train_idx for node in list(cc_list[idx].nodes())]
val_nodes = [node for idx in val_idx for node in list(cc_list[idx].nodes())]
test_nodes = [node for idx in test_idx for node in list(cc_list[idx].nodes())]

print(len(train_nodes), len(val_nodes), len(test_nodes))

In [ ]:
result_df = final_cancer_driver_df[final_cancer_driver_df['node_id'].isin(train_nodes)]
result_df.label.value_counts()

In [ ]:
result_df = final_cancer_driver_df[final_cancer_driver_df['node_id'].isin(val_nodes)]
result_df.label.value_counts()

In [ ]:
result_df = final_cancer_driver_df[final_cancer_driver_df['node_id'].isin(test_nodes)]
result_df.label.value_counts()